# Phase II - Step 4: Per-alert XAI-Q signals (local ESS / SI / AR)

Turns the dataset-level XAI-Q audit into **per-alert trust signals** that the JEV decision layer (next step) can consume.

For every sampled alert we compute, for the paper RF (50 trees, depth 10):

| signal | meaning | how |
|---|---|---|
| `ESS` | explanation stability | mean top-10 Jaccard between SHAP(x) and SHAP(x + small noise), K perturbations. LIME: stability across re-runs with different seeds |
| `SI` | explanation concentration | share of total \|attribution\| carried by the top-5 features (`SI_entropy` = 1 - normalised entropy) |
| `AR` | analyst alignment | share of \|attribution\| on features that the rubric below marks as meaningful traffic-behaviour evidence |
| model-side | `p_attack`, `entropy`, `tree_std` (disagreement among trees), `flip_rate` (prediction flips under noise) | |

Outputs go to `data/phaseII_outputs/xaiq_*.parquet`. SHAP is computed on a stratified sample of the validation split (to train/calibrate JEV) and of the test split (to evaluate JEV); LIME is computed on a smaller test subset only for the SHAP-vs-LIME comparison.

**The AR rubric is a draft and must be reviewed by a domain expert before the paper reports it.**

## 0. Setup and configuration

In [ ]:
import importlib.util
import subprocess
import sys
import time
import warnings
from pathlib import Path

for pkg, mod in [('shap', 'shap'), ('lime', 'lime'), ('pyarrow', 'pyarrow'), ('scikit-learn', 'sklearn'),
                 ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')
SEED = 42
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

# Static path: WSL Ubuntu view of the D: drive (falls back to the Windows path outside WSL)
ROOT = Path('/mnt/d/ai_security/xai-q-audit-ids') if Path('/mnt/d').exists() else Path(r'D:\ai_security\xai-q-audit-ids')
OUT_DIR = ROOT / 'data' / 'phaseII_outputs'

CAP_PER_CAT = 400    # max alerts per attack_cat per split (stratified; sample_weight restores true prevalence)
K_PERT = 5           # perturbations per alert for SHAP ESS
NOISE = 0.10         # Gaussian noise, as a fraction of each numeric feature's std
TOPK = 10            # top-k features used for Jaccard stability
N_LIME_PER_CAT = 50  # LIME subset size per attack_cat (test split only)
LIME_SAMPLES = 500
LIME_RUNS = 3
LIME_FEATURES = 30
print('shap', shap.__version__, '| out dir:', OUT_DIR, '| exists:', OUT_DIR.is_dir())

## 1. Load the split features saved by notebook 02 and refit the paper RF
The RF is refit from `fit.parquet` (about 1 s) instead of unpickling `base_models.joblib`, so the notebook does not depend on the scikit-learn version that wrote the pickle. Same hyper-parameters and seed as notebook 02.

In [ ]:
fit = pd.read_parquet(OUT_DIR / 'fit.parquet')
val = pd.read_parquet(OUT_DIR / 'val.parquet')
test = pd.read_parquet(OUT_DIR / 'test.parquet')
META = ['label', 'attack_cat']
feature_cols = [c for c in fit.columns if c not in META]
d = len(feature_cols)
print('features:', d, '| fit', fit.shape, '| val', val.shape, '| test', test.shape)

rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=SEED)
rf.fit(fit[feature_cols].values, fit['label'].values)
pt = rf.predict_proba(test[feature_cols].values)[:, 1]
print('RF-paper test F1 @0.5: %.3f | AUC %.3f  (notebook 02: F1 0.860, AUC 0.978)'
      % (f1_score(test['label'], pt >= 0.5), roc_auc_score(test['label'], pt)))

## 2. Stratified alert samples
Up to `CAP_PER_CAT` alerts per attack category from validation and test. Each row carries `sample_weight = category size / sampled size`, so weighted statistics reflect the real traffic mix.

In [ ]:
def stratified_sample(df, cap, seed):
    parts = []
    for _, g in df.groupby('attack_cat'):
        n = min(len(g), cap)
        parts.append(g.sample(n, random_state=seed).assign(sample_weight=len(g) / n))
    out = pd.concat(parts)
    out.index.name = 'row_id'
    return out.sort_index()


val_s = stratified_sample(val, CAP_PER_CAT, SEED)
test_s = stratified_sample(test, CAP_PER_CAT, SEED)
print('val sample', len(val_s), '| test sample', len(test_s))
pd.concat([val.attack_cat.value_counts().rename('val_total'), val_s.attack_cat.value_counts().rename('val_sampled'),
           test.attack_cat.value_counts().rename('test_total'), test_s.attack_cat.value_counts().rename('test_sampled')], axis=1)

## 3. AR rubric (draft - needs expert review)
Weight of each feature as evidence an analyst would accept for a network-intrusion alert:
- **1.0** traffic behaviour: volume, rate, timing, loss, jitter, connection-count (`ct_*`) features, `service=*`, `state=*`.
- **0.5** weaker or dataset-specific evidence: `proto=*`, TTL and TCP-window features (`sttl`, `dttl`, `ct_state_ttl`, `swin`, `dwin`), which are known to be simulator-sensitive.
- **0.0** everything else (e.g. `is_sm_ips_ports`, `is_ftp_login`): flags that should not carry a decision by themselves.

`AR(x) = sum_j w_j * |phi_j| / sum_j |phi_j|`.

In [ ]:
CORE = {'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt',
        'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len',
        'ct_srv_src', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'ct_flw_http_mthd',
        'ct_src_ltm', 'ct_srv_dst', 'ct_ftp_cmd'}
PARTIAL = {'sttl', 'dttl', 'ct_state_ttl', 'swin', 'dwin'}


def rubric_weight(name):
    if name.startswith(('service=', 'state=')) or name in CORE:
        return 1.0
    if name.startswith('proto=') or name in PARTIAL:
        return 0.5
    return 0.0


W = np.array([rubric_weight(c) for c in feature_cols])
print('weight 1.0:', int((W == 1).sum()), '| 0.5:', int((W == 0.5).sum()), '| 0.0:', [c for c, w in zip(feature_cols, W) if w == 0])

## 4. Local signal functions

In [ ]:
explainer = shap.TreeExplainer(rf)


def shap_attack(X):
    # SHAP values of the attack class, shape (n, d), robust to the shap version's output layout
    sv = explainer.shap_values(X, check_additivity=False)
    if isinstance(sv, list):
        return np.asarray(sv[1])
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv


def topk_sets(phi, k=TOPK):
    return [set(r) for r in np.argsort(-np.abs(phi), axis=1)[:, :k]]


def jaccard(a, b):
    return len(a & b) / len(a | b)


def concentration(phi):
    a = np.abs(phi)
    share = a / np.maximum(a.sum(1, keepdims=True), 1e-12)
    top5 = np.sort(share, axis=1)[:, -5:].sum(1)
    ent = -(share * np.log(np.maximum(share, 1e-12))).sum(1) / np.log(share.shape[1])
    return top5, 1 - ent


def alignment(phi):
    a = np.abs(phi)
    return (a * W).sum(1) / np.maximum(a.sum(1), 1e-12)


# numeric features get noise; one-hot / binary columns are left untouched
num_idx = [i for i, c in enumerate(feature_cols) if fit[c].nunique() > 2]
fit_arr = fit[feature_cols].values
std, lo, hi = fit_arr.std(0), fit_arr.min(0), fit_arr.max(0)
rng = np.random.default_rng(SEED)


def perturb(X):
    Xp = X.copy()
    noise = rng.normal(0, 1, (len(X), len(num_idx))) * NOISE * std[num_idx]
    Xp[:, num_idx] = np.clip(X[:, num_idx] + noise, lo[num_idx], hi[num_idx])
    return Xp


def shap_signals(S):
    X = S[feature_cols].values.astype(np.float32)
    p = rf.predict_proba(X)[:, 1]
    pred = (p >= 0.5).astype(int)
    tree_p = np.stack([t.predict_proba(X)[:, 1] for t in rf.estimators_], 1)
    phi = shap_attack(X)
    base_top = topk_sets(phi)
    ess = np.zeros(len(X))
    flips = np.zeros(len(X))
    for _ in range(K_PERT):
        Xp = perturb(X)
        top_p = topk_sets(shap_attack(Xp))
        ess += np.array([jaccard(a, b) for a, b in zip(base_top, top_p)]) / K_PERT
        flips += ((rf.predict_proba(Xp)[:, 1] >= 0.5).astype(int) != pred) / K_PERT
    si, si_ent = concentration(phi)
    top5 = np.argsort(-np.abs(phi), axis=1)[:, :5]
    top_feats = [';'.join(f'{feature_cols[j]}:{phi[i, j]:+.3f}' for j in r) for i, r in enumerate(top5)]
    pc = np.clip(p, 1e-9, 1 - 1e-9)
    out = pd.DataFrame({
        'attack_cat': S['attack_cat'].values, 'label': S['label'].values, 'sample_weight': S['sample_weight'].values,
        'p_attack': p, 'pred': pred, 'correct': (pred == S['label'].values).astype(int),
        'entropy': -(pc * np.log2(pc) + (1 - pc) * np.log2(1 - pc)), 'tree_std': tree_p.std(1),
        'ESS': ess, 'flip_rate': flips, 'SI': si, 'SI_entropy': si_ent, 'AR': alignment(phi), 'top_feats': top_feats,
    }, index=S.index)
    return out, phi

## 5. Compute SHAP signals (validation and test samples)

In [ ]:
t0 = time.time()
sig_val, phi_val = shap_signals(val_s)
print('val  done in %.0fs' % (time.time() - t0))
t0 = time.time()
sig_test, phi_test = shap_signals(test_s)
print('test done in %.0fs' % (time.time() - t0))
sig_test.describe().T[['mean', 'std', 'min', '50%', 'max']].round(3)

## 6. Do the signals carry information about errors?
A useful trust signal must be higher (riskier) on wrong predictions than on right ones. AUROC of each risk score for `error = (pred != label)`, sample-weighted; 0.5 means no information.

In [ ]:
def risk_scores(s):
    return pd.DataFrame({
        'uncertainty (1-max p)': 1 - np.maximum(s.p_attack, 1 - s.p_attack),
        'entropy': s.entropy,
        'tree_std': s.tree_std,
        'flip_rate': s.flip_rate,
        'low ESS (1-ESS)': 1 - s.ESS,
        'low AR (1-AR)': 1 - s.AR,
        'low SI (1-SI)': 1 - s.SI,
    })


rows = {}
for name, s in [('val', sig_val), ('test', sig_test)]:
    err = 1 - s['correct'].values
    rows[name] = {c: roc_auc_score(err, v.values, sample_weight=s.sample_weight.values) for c, v in risk_scores(s).items()}
pd.DataFrame(rows).round(3)

In [ ]:
def wmean(g, c):
    return np.average(g[c], weights=g.sample_weight)


outcome = np.select([(sig_test.label == 1) & (sig_test.pred == 1), (sig_test.label == 0) & (sig_test.pred == 0),
                     (sig_test.label == 0) & (sig_test.pred == 1)], ['TP', 'TN', 'FP'], 'FN')
sig_test['outcome'] = outcome
cols = ['p_attack', 'tree_std', 'ESS', 'flip_rate', 'SI', 'AR']
summary = sig_test.groupby('outcome').apply(lambda g: pd.Series({**{c: wmean(g, c) for c in cols}, 'n_sampled': len(g),
                                                                  'est_alerts': g.sample_weight.sum()}))
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
for ax, c in zip(axes, ['ESS', 'SI', 'AR', 'tree_std']):
    data = [sig_test.loc[sig_test.outcome == o, c] for o in ['TP', 'TN', 'FP', 'FN']]
    ax.boxplot(data, showfliers=False)
    ax.set_xticklabels(['TP', 'TN', 'FP', 'FN'])
    ax.set_title(c)
plt.tight_layout()
plt.savefig(OUT_DIR / 'xaiq_signals_by_outcome.png', dpi=150)
plt.show()

## 7. SHAP vs LIME on a test subset
LIME (`discretize_continuous=True`, one-hot columns treated as categorical, `LIME_FEATURES` features kept) is run `LIME_RUNS` times per alert with different seeds. LIME ESS is the mean pairwise top-k Jaccard across those runs (explainer randomness), SHAP ESS is noise-perturbation stability, so the two ESS values are *different notions of stability* and are reported side by side, not as a like-for-like ranking. `shap_lime_agree` is the top-k Jaccard between SHAP and the first LIME run.

In [ ]:
bg = fit.sample(5000, random_state=SEED)[feature_cols].values
cat_idx = [i for i, c in enumerate(feature_cols) if '=' in c]


def make_lime(seed):
    return LimeTabularExplainer(bg, feature_names=feature_cols, class_names=['normal', 'attack'], categorical_features=cat_idx,
                                mode='classification', discretize_continuous=True, random_state=seed)


lime_explainers = [make_lime(SEED + r) for r in range(LIME_RUNS)]


def lime_vec(expl, x, seed):
    e = expl.explain_instance(x, rf.predict_proba, num_features=LIME_FEATURES, num_samples=LIME_SAMPLES, labels=(1,))
    v = np.zeros(d)
    for j, w in e.as_map()[1]:
        v[j] = w
    return v


lime_rows = test_s.groupby('attack_cat', group_keys=False).apply(lambda g: g.sample(min(len(g), N_LIME_PER_CAT), random_state=SEED))
lime_ids = lime_rows.index
Xl = lime_rows[feature_cols].values.astype(np.float32)
t0 = time.time()
lime_phi = [np.stack([lime_vec(ex, x, s) for x in Xl]) for s, ex in enumerate(lime_explainers)]
print('LIME on %d alerts x %d runs: %.0fs' % (len(Xl), LIME_RUNS, time.time() - t0))

tops = [topk_sets(p) for p in lime_phi]
pairs = [(0, 1), (0, 2), (1, 2)][:max(1, LIME_RUNS * (LIME_RUNS - 1) // 2)]
lime_ess = np.mean([[jaccard(a, b) for a, b in zip(tops[i], tops[j])] for i, j in pairs], axis=0)
si5, si_e = concentration(lime_phi[0])
shap_top = topk_sets(phi_test[sig_test.index.get_indexer(lime_ids)])
lime_sig = pd.DataFrame({'ESS': lime_ess, 'SI': si5, 'SI_entropy': si_e, 'AR': alignment(lime_phi[0]),
                         'shap_lime_agree': [jaccard(a, b) for a, b in zip(shap_top, tops[0])]}, index=lime_ids)
lime_sig = lime_sig.join(test_s[['attack_cat', 'label', 'sample_weight']]).join(sig_test[['correct', 'outcome']])
cmp_cols = ['ESS', 'SI', 'AR']
print('mean SHAP-LIME top-%d agreement: %.3f' % (TOPK, lime_sig.shap_lime_agree.mean()))
pd.DataFrame({'SHAP': sig_test.loc[lime_ids, cmp_cols].mean(), 'LIME': lime_sig[cmp_cols].mean()}).round(3)

## 8. Save signals for the JEV step

In [ ]:
sig_val.to_parquet(OUT_DIR / 'xaiq_val.parquet')
sig_test.drop(columns='outcome').to_parquet(OUT_DIR / 'xaiq_test.parquet')
lime_sig.to_parquet(OUT_DIR / 'xaiq_lime_test.parquet')
np.save(OUT_DIR / 'shap_val_phi.npy', phi_val.astype(np.float32))
np.save(OUT_DIR / 'shap_test_phi.npy', phi_test.astype(np.float32))
print(sorted(p.name for p in OUT_DIR.iterdir()))